In [62]:
import os
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[04/18/26 16:45:05] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=30239;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=475312;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\CENTRAL\central-perm-flow


# Importar nodos

In [63]:
# 2. Importar todas las funciones del archivo nodes.py
import central_perm_flow.pipelines.data_processing.nodes as nodes

# Esto te dará una lista limpia de todo lo definido en ese archivo de nodos
print(dir(nodes))


['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '__warningregistry__', 'check_and_export_duplicates', 'check_dataframe', 'clean_column_names', 'clean_column_objects', 'clean_nulls', 'convert_all_standardized_dates', 'convert_standardized_dates', 'datetime', 'duckdb', 'np', 'numeric_conversion_node', 'pd', 're', 'remove_accents', 'remove_accents_and_special_chars', 'select_columns', 'unicodedata']


In [64]:
# Muestra la ayuda formateada
help(nodes.clean_column_objects)

# O simplemente imprime el texto de la descripción
print(nodes.clean_column_objects.__doc__)

Help on function clean_column_objects in module central_perm_flow.pipelines.data_processing.nodes:

clean_column_objects(df: pandas.DataFrame, email_cols: list = None) -> pandas.DataFrame
    Limpia y estandariza columnas de tipo objeto en el DataFrame.

    Aplica conversión a minúsculas, eliminación de espacios y normalización de
    caracteres. Permite identificar columnas de correo para proteger su formato
    evitando la eliminación de '@' y '.'.

    Args:
        df (pd.DataFrame): DataFrame original con columnas de texto a procesar.
        email_cols (list, optional): Lista de nombres de columnas que deben
            tratarse como correos electrónicos. Por defecto es None.

    Returns:
        pd.DataFrame: DataFrame con las columnas de tipo objeto normalizadas.


Limpia y estandariza columnas de tipo objeto en el DataFrame.

Aplica conversión a minúsculas, eliminación de espacios y normalización de 
caracteres. Permite identificar columnas de correo para proteger su formato

# Fuentes de Información

## Base de Estados de los Estudiantes

In [65]:
# Base de central con fechas de bajas, fechas de graduación
central_estaca = catalog.load('central_estaca')['CENTRAL']

[04/18/26 16:45:06] INFO     Loading data from central_estaca (ExcelDataset)...                ]8;id=672066;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=702969;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

[04/18/26 16:45:10] WARNING  c:\Users\User\miniconda3\envs\central\Lib\site-packages\openpyxl\works ]8;id=637417;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py\warnings.py]8;;\:]8;id=661866;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py#110\110]8;;\
                             heet\_reader.py:329: UserWarning: Unknown extension is not supported                  
                             and will be removed                                                                   
                               warn(msg)                                                                           
                                                                                                                   

### Parámetros

In [66]:
central_col_fechas = ['fecha_de_registro', 'cohorte', 'fecha_de_baja_t', 'fecha_de_baja_d', 'fecha_de_reingreso', 'fecha_grado']
central_col_emails = ['usuario_institucional']
central_col_dd = ['alianza', 'cohorte', 'nivel', 'programa', 'identificacion']
central_col_sort = ['codigo_sis']

In [67]:
# Limpieza de la base de operaciones
central_estaca = nodes.clean_column_names(central_estaca)
central_estaca = nodes.convert_all_standardized_dates(central_estaca, ['fecha_de_registro', 'cohorte', 'fecha_de_baja_t', 'fecha_de_baja_d', 'fecha_de_reingreso', 'fecha_grado'])
central_estaca = nodes.clean_column_objects(central_estaca,central_col_emails)
central_estaca_sd, central_estaca_cd  = nodes.check_and_export_duplicates(central_estaca, subset=central_col_dd, col_ordenar = central_col_sort)


## Calendario Académico

In [68]:
# Calendario Académico
central_calendario = catalog.load("central_calaca")

                    INFO     Loading data from central_calaca (ExcelDataset)...                ]8;id=242238;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=964544;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

### Parámetros

In [69]:
central_calaca_col_fechas = ['fecha_inicio', 'fecha_fin']
central_calaca_col_fechaingreso = 'fecha_ingreso'
central_calaca_col_fecha_inicial_semana = 'fecha_inicio'
central_calaca_col_cohorte_inicial = 'cohorte_inicial'
central_calaca_col_sort = ['cohorte_inicial', 'cohorte_actual', 'fecha_inicio', 'semana']
central_calalaca_student_journey = ['onboarding', 'q1', 'qa']

# Definimos las condiciones
condiciones = [
    (central_calendario_extendido['month'] > 0) & (central_calendario_extendido['month'] <= 2),
    (central_calendario_extendido['month'] > 2) & (central_calendario_extendido['month'] <= 4),
    (central_calendario_extendido['month'] > 4)
]

In [70]:
# limpieza del calendario académico
central_calendario = nodes.clean_column_names(central_calendario)
central_calendario = nodes.convert_all_standardized_dates(central_calendario, date_columns =  central_calaca_col_fechas)
central_calendario = nodes.clean_column_objects(central_calendario)

#Fecha de ingreso: la fecha de inicio del primer periodo académico en el que el estudiante estuvo activo
central_calendario[central_calaca_col_fechaingreso] = (
    central_calendario
    .groupby(central_calaca_col_cohorte_inicial)[central_calaca_col_fecha_inicial_semana]
    .transform('min')
)

# Ordenar el DataFrame por cohorte_inicial, cohorte_actual, bloque y semana
central_calendario_extendido = central_calendario.sort_values(by=central_calaca_col_sort).copy()
# Crear un contador de semanas acumuladas por cada cohorte (fecha_ingreso)
central_calendario_extendido['semana_acumulada'] = central_calendario_extendido.groupby(central_calaca_col_fechaingreso).cumcount() + 1

# 3. Calcular el mes corrido
# Usamos (semana - 1) // 4 + 1 para que las semanas 1, 2, 3 y 4 pertenezcan al Mes 1
central_calendario_extendido['month'] = (central_calendario_extendido['semana_acumulada'] - 1) // 4 + 1

# Formatear el mes corrido como 'm1', 'm2', etc.
central_calendario_extendido['mes_label'] = 'm' + central_calendario_extendido['month'].astype(str)
# Ordenar por fecha de ingreso, periodo inicial, bloque y semana
central_calendario_extendido = central_calendario_extendido.sort_values(by=central_calaca_col_sort).reset_index(drop=True)

# Shift para obtener la fecha de inicio del siguiente bloque
#grupos = [central_calaca_col_cohorte_inicial]
central_calendario_extendido['shifted_fecha_inicio'] = central_calendario_extendido.groupby(central_calaca_col_cohorte_inicial)[central_calaca_col_fecha_inicial_semana].shift(-1)
# Rellenar con la útlima fecha de fin para los casos donde no hay siguiente bloque
central_calendario_extendido['shifted_fecha_inicio'] = (
    central_calendario_extendido['shifted_fecha_inicio'].fillna(
        central_calendario_extendido.groupby(central_calaca_col_cohorte_inicial)['fecha_fin'].transform('max') 
    )
)

# Aplicamos de forma vectorizada (mucho más rápido)
central_calendario_extendido['student_journey'] = np.select(condiciones, central_calalaca_student_journey, default='unknown')

                    WARNING  G:\Unidades compartidas\Alianzas\3.                                    ]8;id=385840;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py\warnings.py]8;;\:]8;id=434935;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py#110\110]8;;\
                             Data\CENTRAL\central-perm-flow\src\central_perm_flow\pipelines\data_pr                
                             ocessing\nodes.py:104: UserWarning: Parsing dates in %d/%m/%Y format                  
                             when dayfirst=False (the default) was specified. Pass `dayfirst=True`                 
                             or specify a format to silence this warning.                                          
                               # 3. Convertir la columna date_column a datetime, manejando tanto                   
                             strings como números de serie de Excel                                                
                                                                                                                   

                    WARNING  G:\Unidades compartidas\Alianzas\3.                                    ]8;id=863571;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py\warnings.py]8;;\:]8;id=872069;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py#110\110]8;;\
                             Data\CENTRAL\central-perm-flow\src\central_perm_flow\pipelines\data_pr                
                             ocessing\nodes.py:104: UserWarning: Parsing dates in %d/%m/%Y format                  
                             when dayfirst=False (the default) was specified. Pass `dayfirst=True`                 
                             or specify a format to silence this warning.                                          
                               # 3. Convertir la columna date_column a datetime, manejando tanto                   
                             strings como números de serie de Excel                                                
                                                                                                                   

In [74]:
import fastparquet

In [78]:
central_calendario_extendido = catalog.load('central_calendario_extendido')

[04/18/26 16:59:10] INFO     Loading data from central_calendario_extendido                    ]8;id=920034;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=284643;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

In [79]:
central_calendario_extendido.columns


Index(['cohorte', 'cohorte_inicial', 'fecha_ingreso', 'cohorte_actual',
       'periodo_raw', 'bloque', 'fecha_inicio', 'fecha_fin',
       'shifted_fecha_inicio', 'semana', 'semana_acumulada', 'month',
       'mes_academico', 'mes_gregoriano', 'student_journey', 'tipo'],
      dtype='object')